# 04. 머신러닝 에이전트 — CSV 기반 자유 ML 워크플로

## 학습 목표

- CSV 데이터를 받아 에이전트가 자유롭게 EDA → 전처리 → 모델 선택 → 학습 → 평가를 수행한다
- NB03의 `run_pandas` 패턴을 확장하여 sklearn을 포함하는 `run_ml_code` 도구를 만든다
- 멀티턴 대화로 반복적 분석과 모델 개선을 수행한다
- v1 미들웨어(ToolRetryMiddleware, ModelCallLimitMiddleware)를 적용한다

## 개요

| 항목 | NB03 (데이터 분석) | NB04 (머신러닝) |
|------|-------------------|----------------|
| **데이터** | 매출 CSV (8행) | 유방암 CSV (569행, 30특성) |
| **도구** | `run_pandas` (pandas + numpy) | `run_ml_code` (pandas + numpy + sklearn) |
| **목적** | 집계, 통계, 추이 분석 | EDA → 전처리 → 모델 학습 → 비교 |
| **패턴** | 코드 실행 + 멀티턴 | 코드 실행 + 멀티턴 + 자유 ML 파이프라인 |

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요"

In [ ]:
# Observability 설정 (선택)
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

## NB03 vs NB04: 도구 확장 패턴

NB03에서는 `run_pandas`로 pandas/numpy 코드를 실행했습니다. NB04에서는 이 패턴을 확장하여 **sklearn까지 네임스페이스에 추가**한 `run_ml_code`를 사용합니다.

```python
# NB03: run_pandas
ns = {"pd": pd, "np": np, "csv_path": csv_path}

# NB04: run_ml_code (sklearn 추가)
ns = {"pd": pd, "np": np, "sklearn": sklearn, "csv_path": csv_path}
```

이렇게 하면 에이전트가 데이터 로드부터 모델 학습까지 **자유롭게** 코드를 작성할 수 있습니다.

In [ ]:
from deepagents.backends import LocalShellBackend

backend = LocalShellBackend(root_dir=".", virtual_mode=True)

## 1단계: CSV 데이터 준비

sklearn의 `load_breast_cancer()` 데이터셋을 CSV로 변환하여 임시 디렉토리에 저장합니다.

| 항목 | 내용 |
|------|------|
| **샘플 수** | 569 |
| **특성 수** | 30 (mean radius, mean texture, ...) |
| **타겟** | 0 (malignant), 1 (benign) — 이진 분류 |
| **용도** | 유방암 진단 |

In [ ]:
import tempfile
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "breast_cancer.csv")
df.to_csv(csv_path, index=False)

print(f"CSV 저장: {csv_path}")
print(f"Shape: {df.shape}")
print(f"타겟 분포: {dict(df['target'].value_counts())}")

## 2단계: 도구 정의 — get_csv_path + run_ml_code

| 도구 | 역할 |
|------|------|
| `get_csv_path` | CSV 파일 경로 반환 |
| `run_ml_code` | pandas + numpy + sklearn 코드를 실행하고 결과 반환 |

> `run_ml_code`는 NB03의 `run_pandas`를 확장한 것입니다. `sklearn` 모듈 전체를 네임스페이스에 추가하여, 에이전트가 `sklearn.ensemble`, `sklearn.model_selection` 등을 자유롭게 사용할 수 있습니다.

In [ ]:
from langchain.tools import tool
import io, contextlib

@tool
def get_csv_path() -> str:
    """분석 대상 CSV 파일의 경로를 반환합니다."""
    return csv_path

@tool
def run_ml_code(code: str) -> str:
    """sklearn/pandas Python 코드를 실행합니다. print()로 결과를 출력하세요."""
    import pandas as pd, numpy as np, sklearn
    buf = io.StringIO()
    ns = {"pd": pd, "np": np, "sklearn": sklearn, "csv_path": csv_path}
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
        return buf.getvalue() or "실행 완료 (출력 없음)"
    except Exception as e:
        return f"오류: {e}"

## 3단계: 에이전트 생성

프롬프트에서 에이전트에게 **자유로운 ML 워크플로**를 지시합니다:
1. CSV 경로 확인 → 2. EDA → 3. 전처리 → 4. 모델 학습/비교 → 5. 최적 모델 추천

| 미들웨어 | 역할 |
|---------|------|
| `ToolRetryMiddleware` | 도구 실패 시 자동 재시도 (최대 2회) |
| `ModelCallLimitMiddleware` | 무한 루프 방지 — 최대 20회 모델 호출 제한 |

In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    ToolRetryMiddleware,
    ModelCallLimitMiddleware,
)
from prompts import ML_AGENT_PROMPT

ml_agent = create_deep_agent(
    model=model,
    tools=[get_csv_path, run_ml_code],
    system_prompt=ML_AGENT_PROMPT,
    backend=backend,
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ModelCallLimitMiddleware(run_limit=20),
    ],
)

## 4단계: EDA 분석 요청

에이전트에게 CSV 데이터를 탐색하고 기본 통계를 분석하도록 요청합니다. 에이전트는 `get_csv_path` → `run_ml_code`를 사용하여 자유롭게 EDA를 수행합니다.

In [ ]:
thread = {"configurable": {"thread_id": "ml-1"}}

response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "CSV 데이터를 탐색하고 기본 통계를 분석해주세요. shape, 타겟 분포, 주요 통계량을 확인하세요."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)

## 5단계: 모델 학습 + 비교 요청

에이전트에게 적절한 모델 3개 이상을 학습하고 교차 검증으로 성능을 비교하도록 요청합니다. 에이전트가 **스스로 알고리즘을 선택**합니다.

In [ ]:
response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "적절한 분류 모델 3개 이상을 학습하고, 5-fold 교차 검증으로 성능을 비교해주세요. 결과를 표로 정리하세요."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)

## 6단계: 멀티턴 후속 — Feature Importance 분석

같은 `thread_id`를 사용하여 이전 대화의 맥락을 유지한 채 후속 분석을 요청합니다.

In [ ]:
response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "최적 모델의 feature importance를 분석해주세요. 상위 10개 특성을 보여주세요."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)

## 7단계: 스트리밍 — 추가 분석

`stream(subgraphs=True)`으로 에이전트의 실행 과정을 실시간으로 관찰합니다. 청크(chunk)를 수집하여 에이전트가 어떤 도구를 호출하는지 확인할 수 있습니다.

In [ ]:
chunks = []
for ns, chunk in ml_agent.stream(
    {"messages": [{"role": "user", "content": "데이터에 스케일링을 적용한 뒤 동일한 모델들을 다시 비교해주세요. 스케일링 전후 성능 차이를 표로 정리하세요."}]},
    config={**thread, **lf_config},
    subgraphs=True,
):
    chunks.append((ns, chunk))

print(f"총 {len(chunks)}개 청크 수신", flush=True)
last = chunks[-1][1] if chunks else {}
if hasattr(last, "get") and last.get("messages"):
    print(last["messages"][-1].content, flush=True)

## 요약

| 항목 | 핵심 |
|------|------|
| **데이터** | 유방암 CSV (569샘플, 30특성, 이진 분류) |
| **도구** | `get_csv_path` + `run_ml_code` (pandas + numpy + sklearn) |
| **워크플로** | 자유 EDA → 전처리 → 모델 선택 → 교차 검증 비교 |
| **멀티턴** | `InMemorySaver` + 동일 `thread_id` — 대화 맥락 유지 |
| **미들웨어** | `ToolRetryMiddleware` (재시도) + `ModelCallLimitMiddleware` (무한루프 방지) |

---

**참고 문서:**
- `docs/deepagents/06-backends.md`
- `docs/deepagents/tutorials/data-analysis.md`
- [scikit-learn 공식 문서](https://scikit-learn.org/stable/)

**다음 단계:** → [05_deep_research_agent.ipynb](./05_deep_research_agent.ipynb): 딥 리서치 에이전트를 구축합니다.